In [0]:
from datetime import date, timedelta
from pyspark.sql.functions import col, count, sum as _sum

dbutils.widgets.dropdown(
    "time_period",
    "daily",
    ["daily", "weekly", "monthly"]
)

time_period = dbutils.widgets.get("time_period")
report_date = date(2024, 12, 12)   # fixed to dataset range



In [0]:
from pyspark.sql.functions import to_date, col


if time_period == "daily":
    start_date = report_date - timedelta(days=1)
    end_date = start_date

elif time_period == "weekly":
    start_date = report_date - timedelta(days=report_date.weekday() + 7)
    end_date = start_date + timedelta(days=6)

elif time_period == "monthly":
    first_day_current_month = report_date.replace(day=1)
    end_date = first_day_current_month - timedelta(days=1)
    start_date = end_date.replace(day=1)

orders_df = (
    spark.read.parquet("/Volumes/workspace/default/shree_project/silver/orders")
    .withColumn(
        "order_date_parsed",
        to_date(col("order_date"), "dd-MM-yyyy")
    )
)

# from pyspark.sql.functions import expr

# orders_df = orders_df.withColumn(
#     "order_date_parsed",
#     expr("try_to_date(order_date, 'dd-MM-yyyy')")
# )


customers_df = spark.read.parquet(
    "/Volumes/workspace/default/shree_project/silver/customers"
)



In [0]:
report_df = (
    orders_df
    .join(customers_df, "customer_id", "inner")
    .filter(
        (col("order_date_parsed") >= start_date) &
        (col("order_date_parsed") <= end_date)
    )
    .groupBy("city")
    .agg(
        count("order_id").alias("total_orders"),
        _sum("quantity").alias("total_quantity")
    )
)



print(f"Report Type : {time_period.upper()}")
print(f"Data Range  : {start_date} to {end_date}")

display(report_df)


